In [720]:

import os
import numpy as np
from sklearn.model_selection import train_test_split
import pathlib


def load_data(datatype):

    X_data = []
    y_data = []

    # Directory path to the data
    path = os.path.join(pathlib.Path().resolve(),datatype)
    directory_path = os.path.abspath(path)

    #print(directory_path)

    for filename in glob.glob(os.path.join(directory_path, '*.csv')):

        # print(filename.rindex('_0'))
        # print(filename[filename.rindex('_0')-1])

        # Extracting the class labels from the filenames
        y = int(filename[filename.rindex('_0')-1])
        y_data.append(y)

        # Read the csv data file and convert into numpy array
        x = np.genfromtxt(filename, delimiter=',')
        X_data.append(x)


    # X_data = np.array(X_data[0])
    # X_data
    # print(X_data.shape)

    # Checking classes and their counts
    values, counts = np.unique(y_data, return_counts=True)

    print(f'The data has {len(X_data)} samples with {values} classes and respectively their counts {counts}.')

    return X_data, y_data


# Loading training data from the 
X_train, y_train = load_data('training_data')  

C:\Users\jonna\OneDrive\Documents\Pattern recognition and machine learning 2025\training_data
The data has 1000 samples with [0 1 2 3 4 5 6 7 8 9] classes and respectively their counts [100 100 100 100 100 100 100 100 100 100].


In [721]:
def standardize_data(X_data):

    X_data_std = [] # Standardized data

    for i in range(len(X_data)):


        X = X_data[i]
    

        # Standardization Z = (X-µ) / σ for each column
        mu = X.mean(axis=0)
        sigma = X.std(axis=0, ddof=1) # Sample standard deviation
        Z = (X - mu) / sigma
        X_data_std.append(Z)

    return X_data_std # Returns the standardized data as a numpy array
    
X_train = standardize_data(X_data=X_train)

In [722]:
def pad_shape(X_train):          # makin all the data matrices same size by addind zeros
    max_rows = max(x.shape[0] for x in X_train)
    max_cols = max(x.shape[1] for x in X_train)
    padded = []
    for x in X_train:
        pad_rows = max_rows - x.shape[0]
        pad_cols = max_cols - x.shape[1]
        padded_x = np.pad(x, ((0, pad_rows), (0, pad_cols)), mode='constant')
        padded.append(padded_x)
    return np.array(padded)

def flatten(X):       # making all the data matrices into vectors
    f = []
    for i in X:
        flat=np.array(i).flatten()
        f.append(flat)
    return np.array(f)

X_padded = pad_shape(X_train)
flatX = flatten(X_padded)

In [723]:
X_train, X_test, y_train, y_test = train_test_split(flatX, y_train, test_size=.30, random_state=0, shuffle=True)

In [724]:
class NN:
    def __init__(self,dim): #activation and weight initialization
        self.size = dim
        self.activation = self.sigmoid
        self.params = self.initialize()
        self.cache = {}     # for saving all intermediate values

    def sigmoid(self, z):  # Sigmoid activation function
        return 1 / (1 + np.exp(-z)) # gets warning from overflow, with changes in back propagation the warning will most come from the sigmoid_derivate

    def sigmoid_derivate(self,a):
        return (np.exp(-a))/((np.exp(-a)+1)**2)

    def initialize(self):   # initializing weights and bias
        input_layer = self.size  # size of the input layer
        hidden_layer = 64   # size of hidden layer (just randomly picked for now, can be changed)
        output_layer = 10    # size of the output layer, number of different outputs
        
        params = {
            "W": np.random.randn(hidden_layer, input_layer) * np.sqrt(1./input_layer),     # weights 
            "b": np.zeros((hidden_layer, 1)) * np.sqrt(1./input_layer)             # bias
        }
        return params

    def forward_propagation(self, inputs):   # forward propagation
        self.cache["X"] = inputs
        self.cache["Z"] = np.matmul(self.params["W"], self.cache["X"].T) + self.params["b"] # calculating the matrix product between weights and inputs, adding bias
        self.cache["A"] = self.activation(self.cache["Z"])
        return self.cache["A"]

    def back_propagation(self, y, output):  # back propagation, stochastic gradient descent
        y_size = y.shape[0]
        dZ = output - y.T
        #dA = np.matmul(self.params["W"].T, dZ)               # These few lines of code should be used
        #dZ = dA * self.sigmoid_derivate(self.cache["Z"])      
        #dW = (1./y_size) * np.matmul(dZ, self.cache["X"])     
        #db = (1./y_size) * np.sum(dZ, axis=1, keepdims=True)     # but for them to work, y need to be one-hot encoded

        dW = (1./y_size) * np.matmul(dZ, self.cache["X"])   # new weight
        db = (1./y_size) * np.sum(dZ, axis=1, keepdims=True)    # new bias
        self.grads = {"W": dW, "b": db}  
        return self.grads
        
    def optimize(self, l_rate):  # updating parameters
        for key in self.params:
            self.params[key] = self.params[key] - l_rate * self.grads[key]

    def cross_entropy_loss(self, train_outputs, output):  # loss function, cross entropy
        loss_sum = np.sum(np.multiply(train_outputs.T, np.log(output)))  # warning from division by zero in log(which can be fixed) and invalid value encountered in multiply
        m = y.shape[0]
        loss = -(1./m) * loss_sum
        return loss    

    def train(self, train_inputs, train_outputs, epochs, learning_rate=0.1, verbose=False):
        for iteration in range(epochs):
            output = self.forward_propagation(train_inputs) # forward propagation
            grad = self.back_propagation(train_outputs, output) # back propagation, returns gradients of weights
            self.optimize(l_rate=learning_rate)  # updating parameters

            if verbose and (iteration % max(1, epochs // 10) == 0):  # to check on progress
                loss = self.cross_entropy_loss(train_outputs, output)
                train_acc = self.accuracy(train_outputs, output)
                print(f"Iter {iteration:6d}  loss={loss:.6f}")
                print(f"Accuracy={train_acc:.6f}")   
        
    def accuracy(self, train_outputs, output):  # calculating accuracy
        return np.mean(np.argmax(train_outputs, axis=-1) == np.argmax(output.T, axis=-1))


    #def test(self, x, y):
        # to be added later if needed

In [725]:
dim = X_train.shape[1]
nn = NN(dim)

# changing y_train to work for current code
y_train = np.array(y_train)     # one-hot encoding would be needed for to make the code better 
y_train = y_train.reshape(700,1)   # change to be done later if needed
    
nn.train(X_train, y_train, epochs=100, learning_rate=0.2, verbose=True)

Iter      0  loss=35988.212277
Accuracy=0.011429
Iter     10  loss=12356.462774
Accuracy=0.901429
Iter     20  loss=24310.772699
Accuracy=0.955714
Iter     30  loss=36299.407565
Accuracy=0.965714
Iter     40  loss=48298.882920
Accuracy=0.970000
Iter     50  loss=60302.082566
Accuracy=0.971429
Iter     60  loss=72306.318099
Accuracy=0.971429
Iter     70  loss=84310.597622
Accuracy=0.972857
Iter     80  loss=96314.571641
Accuracy=0.972857
Iter     90  loss=108318.130420
Accuracy=0.975714
